# Stream on Colab (T4 GPU)
Byte-level SSM language model â€” token-free, position-free, O(n).

**What this notebook does:**
1. Clones the repo
2. Installs dependencies (inc. `torch.compile` + Triton support)
3. Downloads TinyStories â†’ byte-level dataset
4. Verifies model loads + runs
5. Benchmarks Stream vs GPT on GPU
6. Runs scaling curve (4 sizes)
7. Runs training (configurable)
8. Saves results to Google Drive

**Runtime:** Runtime â†’ Change runtime type â†’ T4 GPU

In [ ]:
# @title 1. Mount Drive & Clone Repo
import os, sys

# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/nishantXnova/RETRANS-X.git'
PROJECT_DIR = '/content/RETRANS-X'
DRIVE_DIR = '/content/drive/MyDrive/stream_results'

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print('Repo already cloned, pulling latest...')
    %cd {PROJECT_DIR}
    !git pull

%cd {PROJECT_DIR}
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'\nProject at: {PROJECT_DIR}')
print(f'Results at: {DRIVE_DIR}')

In [ ]:
# @title 2. Install Dependencies
import torch
print(f'PyTorch {torch.__version__}, CUDA {torch.version.cuda}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB')
print(f'Compute Capability: {torch.cuda.get_device_capability(0)}')

# Check if triton is available
try:
    import triton
    print(f'Triton {triton.__version__} available â€” torch.compile will work!')
except:
    print('Installing triton...')
    !pip install -q triton
    import triton
    print(f'Triton {triton.__version__} installed')

# Verify torch.compile works
def test_fn(x): return x * 2
compiled = torch.compile(test_fn)
_ = compiled(torch.tensor([1.0], device='cuda'))
print('torch.compile() works!')

# Try installing mamba-ssm (optional â€” may have build issues)
try:
    !pip install -q mamba-ssm 2>&1 | tail -5
    from mamba_ssm import Mamba
    print('mamba-ssm installed!')
except:
    print('mamba-ssm not installed (optional, not needed for Stream)')

In [ ]:
# @title 3. Prepare Data
import sys
sys.path.insert(0, PROJECT_DIR + '/VECTOR')
from data.prepare_bytes import prepare_bytes

DATA_DIR = 'VECTOR/data/bytes'
TRAIN_FILE = os.path.join(DATA_DIR, 'train.bin')

if not os.path.exists(TRAIN_FILE):
    print('Creating byte-level dataset from TinyStories...')
    tinystories_url = 'https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStories-train.txt'
    tinystories_path = os.path.join(DATA_DIR, 'TinyStories-train.txt')

    if not os.path.exists(tinystories_path):
        print('Downloading TinyStories (940MB)...')
        import requests
        r = requests.get(tinystories_url, stream=True)
        with open(tinystories_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        print('Download complete.')

    prepare_bytes(tinystories_path)
else:
    print('Data already prepared.')

# Check data
train_size = os.path.getsize(TRAIN_FILE)
val_size = os.path.getsize('VECTOR/data/bytes/val.bin')
print(f'Train: {train_size/1e6:.0f} MB, Val: {val_size/1e6:.0f} MB')

In [ ]:
# @title 4. Verify Model Loads
sys.path.insert(0, PROJECT_DIR + '/VECTOR')
from model import Stream, StreamConfig

device = 'cuda'
model = Stream(StreamConfig(
    n_embd=128, n_layer=2, ssm_d_state=8,
    n_predict=4, block_size=256
)).to(device)

x = torch.randint(0, 256, (1, 256), device=device)
logits, loss = model(x, targets=x)
print(f'Forward OK, loss = {loss.item():.4f}')

loss.backward()
print('Backward OK')
print(f'Parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M')

# Test torch.compile on full model
print('\nCompiling model with torch.compile...')
model_compiled = torch.compile(model)
x = torch.randint(0, 256, (1, 256), device=device)
logits, loss = model_compiled(x, targets=x)
print(f'Compiled forward OK, loss = {loss.item():.4f}')
loss.backward()
print('Compiled backward OK')

In [ ]:
# @title 5. GPU Benchmark: Stream vs GPT
import sys, time, numpy as np, torch

sys.path.insert(0, PROJECT_DIR + '/VECTOR')
from model import Stream, StreamConfig

# Import nanoGPT
import importlib.util
spec = importlib.util.spec_from_file_location('ngpt', 'nanoGPT/model.py')
ngpt = importlib.util.module_from_spec(spec)
sys.modules['ngpt'] = ngpt
spec.loader.exec_module(ngpt)

device = 'cuda'
TS = [512, 1024, 2048, 4096, 8192]

models = [
    ('Stream 4L/128D',  lambda T: Stream(StreamConfig(n_embd=128, n_layer=4, ssm_d_state=8, block_size=T)).to(device).eval()),
    ('GPT 3L/128D',     lambda T: ngpt.GPT(ngpt.GPTConfig(vocab_size=256, n_embd=128, n_layer=3, n_head=4, block_size=T, dropout=0.0, bias=False)).to(device).eval()),
    ('Stream 6L/256D',  lambda T: Stream(StreamConfig(n_embd=256, n_layer=6, ssm_d_state=16, block_size=T)).to(device).eval()),
    ('GPT 8L/192D',     lambda T: ngpt.GPT(ngpt.GPTConfig(vocab_size=256, n_embd=192, n_layer=8, n_head=6, block_size=T, dropout=0.0, bias=False)).to(device).eval()),
]

# Also benchmark with torch.compile
models_compiled = [
    ('Stream 4L/128D (compiled)',  lambda T: torch.compile(Stream(StreamConfig(n_embd=128, n_layer=4, ssm_d_state=8, block_size=T)).to(device).eval())),
    ('Stream 6L/256D (compiled)',  lambda T: torch.compile(Stream(StreamConfig(n_embd=256, n_layer=6, ssm_d_state=16, block_size=T)).to(device).eval())),
]

results = {}
for name, make_fn in models + models_compiled:
    print(f'\n--- {name} ---')
    for B in [1, 4]:
        for T in TS:
            try:
                model = make_fn(T)
                x = torch.randint(0, 256, (B, min(T, model.config.block_size-1 if hasattr(model, 'config') else T)), device=device)
                with torch.no_grad():
                    for _ in range(5): model(x)
                    times = []
                    n = 15 if T <= 2048 else 8
                    for _ in range(n):
                        torch.cuda.synchronize()
                        t0 = time.perf_counter()
                        model(x)
                        torch.cuda.synchronize()
                        times.append(time.perf_counter() - t0)
                fwd_ms = np.median(times) * 1000
                print(f'  T={T:>5}, B={B}: {fwd_ms:>8.2f} ms')
                results[(name, T, B)] = fwd_ms
                del model
                torch.cuda.empty_cache()
            except Exception as e:
                print(f'  T={T:>5}, B={B}: ERROR: {str(e)[:80]}')

# Save results
import pickle
with open(os.path.join(DRIVE_DIR, 'gpu_bench_results.pkl'), 'wb') as f:
    pickle.dump(results, f)
print(f'\nResults saved to {DRIVE_DIR}/gpu_bench_results.pkl')

In [ ]:
# @title 6. Scaling Curve (4 sizes)
import sys, time, numpy as np, torch
sys.path.insert(0, PROJECT_DIR + '/VECTOR')

device = 'cuda'
T = 256
batch_size = 16

# Load data
data = np.memmap('VECTOR/data/bytes/train.bin', dtype=np.uint8, mode='r')

configs = [
    ('tiny',   dict(n_embd=64,  n_layer=2, ssm_d_state=4)),
    ('small',  dict(n_embd=128, n_layer=4, ssm_d_state=8)),
    ('medium', dict(n_embd=192, n_layer=6, ssm_d_state=12)),
    ('large',  dict(n_embd=256, n_layer=8, ssm_d_state=16)),
]

from model import Stream, StreamConfig

scale_results = {}
for name, cfg in configs:
    print(f'\n--- {name} (n_embd={cfg["n_embd"]}, n_layer={cfg["n_layer"]}) ---')
    model = Stream(StreamConfig(n_predict=4, block_size=T, **cfg)).to(device)
    model.train()
    params = sum(p.numel() for p in model.parameters())
    print(f'  Parameters: {params/1e6:.2f}M')

    # Measure tokens/sec
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
    ix = torch.randint(len(data) - T, (batch_size,))
    x = torch.stack([torch.from_numpy(data[i:i+T].astype(np.int64)) for i in ix]).to(device)
    y = x.clone()

    # Warmup
    for _ in range(10):
        _, loss = model(x, targets=y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    # Measured
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(50):
        _, loss = model(x, targets=y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    tok_s = batch_size * T * 50 / elapsed

    print(f'  Throughput: {tok_s:.0f} tokens/sec')
    scale_results[name] = {'params': params, 'tok_s': tok_s, 'val_loss': loss.item()}

print('\n=== Scaling Summary ===')
for name, r in scale_results.items():
    print(f'{name:<10}: {r["params"]/1e6:.2f}M params, {r["tok_s"]:.0f} tok/s, val_loss={r["val_loss"]:.4f}')

with open(os.path.join(DRIVE_DIR, 'scaling_results.pkl'), 'wb') as f:
    pickle.dump(scale_results, f)
print(f'Results saved to {DRIVE_DIR}/scaling_results.pkl')

In [ ]:
# @title 7. Training (configurable)
import sys, os
sys.path.insert(0, PROJECT_DIR + '/VECTOR')

# Pick a config
CONFIG = 'stream_gpu_4k'  # or stream_gpu, stream_light, etc.

print(f'Running config: {CONFIG}')
print(f'Command: python train.py config/{CONFIG}.py')

# The train script uses configurator.py which exec's the config file
# We run it as a subprocess to get clean stdout
!cd VECTOR && python train.py config/{CONFIG}.py 2>&1 | tee {DRIVE_DIR}/train_{CONFIG}.log

# Check if checkpoint was saved
import glob
ckpts = glob.glob('VECTOR/out*/ckpt.pt')
for ckpt in ckpts:
    import shutil
    dest = os.path.join(DRIVE_DIR, os.path.basename(os.path.dirname(ckpt)) + '_ckpt.pt')
    shutil.copy(ckpt, dest)
    print(f'Copied {ckpt} â†’ {dest}')

In [ ]:
# @title 8. Download Results to Local Machine
import os

DRIVE_DIR = '/content/drive/MyDrive/stream_results'

print(f'Results are in: {DRIVE_DIR}')
print('\nFiles:')
for f in os.listdir(DRIVE_DIR):
    size = os.path.getsize(os.path.join(DRIVE_DIR, f))
    print(f'  {f}  ({size/1e3:.0f} KB)')

# zip for easy download
import shutil
shutil.make_archive('/content/stream_results', 'zip', DRIVE_DIR)
print('\nResults also zipped to: /content/stream_results.zip')
print('Download via: Files sidebar â†’ stream_results.zip â†’ Download')